# Train DataLoader 검증 노트북 (Colab 완결판)

`new_dataset.py` / `new_datamodule.py` 검증용. **셀 순서대로** 실행하세요.

| Step | 내용 | 필수 |
|------|------|------|
| **Colab** | Drive 마운트 + `git clone`/`git pull` + pip | Colab만 |
| 0 | 환경 설정 (`data/` → Drive) | ✓ |
| helpers | 다운로드/압축 유틸 | ✓ |
| 1 | Zenodo dev set → `data/dev_set/` | dataloader 스모크 시 |
| 2 | SpAudSyn | dataloader 스모크 시 |
| 3 | Semantic Hearing 다운로드 확인 (tar 압축 해제) | 선택 |
| 3b | EARS 다운로드 | 선택 |
| 4 | `add_data.sh` 실행 → dev_set 보강 | 선택 |
| 4-debug | SEMHEAR_DIR 내부 구조 확인 | 문제 발생 시 |
| 5 | 경로 검증 | dataloader 스모크 시 |
| **unittest** | **`test_new_*.py` 전용 셀** | ✓ (데이터 불필요) |
| 7–8 | train/val dataloader 스모크 | Step 1–2·5 후 |

> **baseline 데이터 준비 순서**: Colab → 0 → helpers → 1 → 2 → 3 → 3b → 4 → 5 → unittest → 7 → 8
>
> **baseline 공식**: `bash add_data.sh --semhear_path /path/to/BinauralCuratedDataset --ears_path /path/to/EARS`
>
> **unittest만**: Colab → 0 → helpers → **unittest** 셀 (Step 1–5 skip 가능)

## [Colab] Drive 마운트 + repo clone / pull

로컬 PC면 **skip**. Repo 최신 반영: `git pull`

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Mockdd/dcase2026_task4.git"
COLAB_REPO_DIR = Path("/content/dcase2026_task4")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/dcase2026_task4/data")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

    if (COLAB_REPO_DIR / ".git").exists():
        print("[git pull]")
        subprocess.run(["git", "pull"], cwd=COLAB_REPO_DIR, check=True)
    elif not (COLAB_REPO_DIR / "src" / "datamodules" / "new_dataset.py").exists():
        print("[git clone]", REPO_URL)
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)

    %cd {COLAB_REPO_DIR}
    !pip install -q lightning pyyaml soundfile librosa scipy python-sofa tqdm
    !git log -1 --oneline
    print("\nColab OK | code:", COLAB_REPO_DIR)
    print("         | data:", DRIVE_DATA_DIR)
else:
    print("Not Colab — skip this cell.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[git pull]
/content/dcase2026_task4
49d76b5 (HEAD -> main, origin/main, origin/HEAD) unittest wrong expectation

Colab OK | code: /content/dcase2026_task4
         | data: /content/drive/MyDrive/dcase2026_task4/data


## Step 0. 환경 설정

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import tarfile
import unittest
import urllib.request
import zipfile
from io import StringIO
from pathlib import Path

import torch
import yaml

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _data_is_scaffold_only(data_path: Path) -> bool:
    if not data_path.exists():
        return True
    return not any(p.is_file() and p.stat().st_size > 0 for p in data_path.rglob("*"))


if IN_COLAB:
    PROJECT_ROOT = Path("/content/dcase2026_task4")
    DATA_DIR = Path("/content/drive/MyDrive/dcase2026_task4/data")
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src" / "datamodules" / "new_dataset.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

LOCAL_DATA = PROJECT_ROOT / "data"
if IN_COLAB:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if LOCAL_DATA.is_symlink() and LOCAL_DATA.resolve() == DATA_DIR.resolve():
        print("[ok] symlink already set")
    elif _data_is_scaffold_only(LOCAL_DATA):
        if LOCAL_DATA.exists() or LOCAL_DATA.is_symlink():
            LOCAL_DATA.unlink() if LOCAL_DATA.is_symlink() else shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] symlink created")
    else:
        shutil.copytree(LOCAL_DATA, DATA_DIR, dirs_exist_ok=True)
        shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] local data moved to Drive + symlink")
    print("symlink:", LOCAL_DATA, "->", DATA_DIR.resolve())

DEV_SET_DIR = DATA_DIR / "dev_set"
DOWNLOAD_DIR = DATA_DIR / "downloads"
ZENODO_DIR = DOWNLOAD_DIR / "zenodo"
SEMHEAR_TAR = DATA_DIR / "BinauralCuratedDataset.tar"
SEMHEAR_DIR = DATA_DIR / "BinauralCuratedDataset"
SPAUDSYN_DIR = PROJECT_ROOT / "src" / "modules" / "spatial_audio_synthesizer"

for d in (DATA_DIR, DOWNLOAD_DIR, ZENODO_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_DIR    :", DATA_DIR.resolve())
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

[ok] local data moved to Drive + symlink
symlink: /content/dcase2026_task4/data -> /content/drive/MyDrive/dcase2026_task4/data
PROJECT_ROOT: /content/dcase2026_task4
DATA_DIR    : /content/drive/MyDrive/dcase2026_task4/data
torch: 2.11.0+cu128 | cuda: True


## helpers. 다운로드 / Zenodo split-zip 유틸

In [ ]:
def describe_batch(batch, title="batch"):
    print(f"\n=== {title} ===")
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k:20s} shape={tuple(v.shape)} dtype={v.dtype}")
        elif isinstance(v, list):
            print(f"  {k:20s} list[len={len(v)}]")
        else:
            print(f"  {k:20s} {type(v).__name__}")


def download_url(url: str, dest: Path, label: str = "") -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {label or dest.name}")
        return
    print(f"[download] {label or dest.name}")
    def _progress(n, size, total):
        if total > 0:
            print(f"\r  {min(n*size,total)/total*100:5.1f}%", end="", flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print("\n  done.")


def _ensure_archive_tools():
    pkgs = [("unzip", "unzip"), ("zip", "zip"), ("p7zip-full", "7z")]
    need = [pkg for pkg, cmd in pkgs if not shutil.which(cmd)]
    if need:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", *need], check=True)


def _unzip_test(path: Path) -> bool:
    return subprocess.run(["unzip", "-t", str(path)], capture_output=True).returncode == 0


def _extract_with_unzip(zip_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-o", "-q", str(zip_path), "-d", str(dest_dir)], check=True)


def _extract_with_7z(zip_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["7z", "x", "-y", f"-o{dest_dir}", zip_path.name], cwd=zip_path.parent, check=True)


def prepare_zenodo_dev_set(zenodo_dir: Path, extract_root: Path) -> None:
    """Zenodo split zip → dev_set. zip -s 0 / unzip / 7z fallback. Python zipfile 사용 안 함."""
    if (DEV_SET_DIR / "metadata" / "valid.json").exists():
        print("[skip] dev_set already installed")
        return

    _ensure_archive_tools()
    split_zip = zenodo_dir / "DCASE2026Task4Dataset.zip"
    merged_zip = zenodo_dir / "DCASE2026Task4DatasetFull.zip"
    if not split_zip.exists():
        raise FileNotFoundError(f"Missing {split_zip}")

    if merged_zip.exists() and not _unzip_test(merged_zip):
        print(f"[cleanup] bad merged zip: {merged_zip.name}")
        merged_zip.unlink()

    marker = extract_root / ".extracted"
    if marker.exists() and not any(extract_root.rglob("**/valid.json")):
        marker.unlink()
    if extract_root.exists() and not any(extract_root.rglob("**/valid.json")):
        shutil.rmtree(extract_root)

    if _extract_done(extract_root):
        print("[skip] extract folder ready")
    else:
        extract_root.mkdir(parents=True, exist_ok=True)
        ok = False
        print("[extract A] unzip split archive")
        try:
            _extract_with_unzip(split_zip, extract_root)
            ok = _extract_done(extract_root)
        except subprocess.CalledProcessError as e:
            print("  failed:", e)
        if not ok:
            if not merged_zip.exists():
                print("[merge] zip -s 0")
                subprocess.run(
                    ["zip", "-s", "0", split_zip.name, "--out", merged_zip.name],
                    cwd=zenodo_dir, check=True,
                )
            print("[extract B] unzip merged")
            _extract_with_unzip(merged_zip, extract_root)
            ok = _extract_done(extract_root)
        if not ok:
            print("[extract C] 7z")
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True)
            _extract_with_7z(split_zip, extract_root)
            ok = _extract_done(extract_root)
        if not ok:
            raise RuntimeError("Zenodo extract failed — check .z01-.z10 downloads")
        marker.write_text("ok", encoding="utf-8")

    install_dev_set_from_extract(extract_root)


def _extract_done(extract_root: Path) -> bool:
    return any(extract_root.rglob("**/valid.json")) if extract_root.exists() else False


def install_dev_set_from_extract(extract_root: Path) -> None:
    if (DEV_SET_DIR / "metadata" / "valid.json").exists():
        return
    candidates = [
        extract_root / "DCASE2026Task4Dataset" / "data" / "dev_set",
        extract_root / "data" / "dev_set",
    ]
    candidates += [p for p in extract_root.rglob("dev_set") if (p / "metadata" / "valid.json").exists()]
    src = next((p for p in candidates if p.exists()), None)
    if src is None:
        raise FileNotFoundError(f"dev_set not found under {extract_root}")
    print(f"[install] {src} -> {DEV_SET_DIR}")
    if DEV_SET_DIR.exists():
        shutil.rmtree(DEV_SET_DIR)
    shutil.copytree(src, DEV_SET_DIR)

def missing_from_config(input_dir: Path, pickup_json: Path) -> list:
    """pickup_json(file_name 리스트) 대비 input_dir에 실제로 없는 wav 목록 반환."""
    import json as _json
    if not pickup_json.exists():
        raise FileNotFoundError(f"pickup_json 없음: {pickup_json} (Step 1 dev_set 필요)")
    with open(pickup_json) as f:
        rows = _json.load(f)
    return [r["file_name"] for r in rows if not (input_dir / r["file_name"]).exists()]


def verify_dataset_complete(input_dir: Path, pickup_json: Path, label: str, strict: bool = False) -> list:
    """input_dir이 pickup_json의 모든 파일을 갖췄는지 검증.
    누락 목록 반환. strict=True면 누락 시 예외, False면 경고만."""
    missing = missing_from_config(input_dir, pickup_json)
    if missing:
        msg = f"{label}: {len(missing)}개 wav 누락 (예: {missing[:3]})"
        if strict:
            raise RuntimeError(f"{label} 불완전 — {input_dir} 에 {len(missing)}개 누락. 데이터 다운로드 필요.")
        print(f"[warn] {msg} — 있는 파일만으로 진행")
    else:
        print(f"[OK] {label}: pickup_json의 모든 wav 존재 ({input_dir})")
    return missing

print("helpers loaded.")

helpers loaded.


## Step 1. Zenodo DCASE dev set

[Zenodo 19328046](https://zenodo.org/records/19328046) — Drive `data/downloads/zenodo/`에 저장

In [ ]:
ZENODO_URLS = [
    line.strip()
    for line in (PROJECT_ROOT / "dev_set_zenodo.txt").read_text(encoding="utf-8").splitlines()
    if line.strip() and "lisence.pdf" not in line.strip().lower()
]
for url in ZENODO_URLS:
    fname = url.rsplit("/", 1)[-1]
    download_url(url, ZENODO_DIR / fname, label=fname)

extract_root = ZENODO_DIR / "extracted"
prepare_zenodo_dev_set(ZENODO_DIR, extract_root)
assert (DEV_SET_DIR / "metadata" / "valid.json").exists()
print("\nStep 1 OK:", DEV_SET_DIR)

## Step 2. SpAudSyn

In [ ]:
marker = SPAUDSYN_DIR / "spatial_audio_synthesizer.py"
if marker.exists():
    print(f"[skip] {SPAUDSYN_DIR}")
else:
    clone_dir = DOWNLOAD_DIR / "SpAudSyn_repo"
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nttcslab/SpAudSyn.git", str(clone_dir)], check=True)
    SPAUDSYN_DIR.parent.mkdir(parents=True, exist_ok=True)
    if SPAUDSYN_DIR.exists():
        shutil.rmtree(SPAUDSYN_DIR)
    shutil.copytree(clone_dir / "src", SPAUDSYN_DIR)
    print(f"[install] {SPAUDSYN_DIR}")
assert marker.exists()
print("\nStep 2 OK")

[install] /content/dcase2026_task4/src/modules/spatial_audio_synthesizer

Step 2 OK


## Step 3. Semantic Hearing 압축 해제 확인

[SemanticHearing](https://github.com/vb000/SemanticHearing) — `BinauralCuratedDataset.tar`를 Drive에 미리 다운로드해두세요.

```
wget -P data https://semantichearing.cs.washington.edu/BinauralCuratedDataset.tar
```

이 셀은 tar가 이미 Drive에 있으면 압축 해제하고, 없으면 다운로드 후 해제합니다.

In [ ]:
# Step 3: Semantic Hearing tar 압축 해제 + FSD50K 완전성 검증
import tarfile

SEMHEAR_URL = "https://semantichearing.cs.washington.edu/BinauralCuratedDataset.tar"
FSD50K_CONFIG = DEV_SET_DIR / "config" / "FSD50K_config.json"


def _extract_semhear_tar():
    if not (SEMHEAR_TAR.exists() and SEMHEAR_TAR.stat().st_size > 0):
        print(f"[download] {SEMHEAR_URL}")
        download_url(SEMHEAR_URL, SEMHEAR_TAR, label="BinauralCuratedDataset.tar")
    print(f"[extract] {SEMHEAR_TAR} → {DATA_DIR}")
    with tarfile.open(SEMHEAR_TAR, "r") as tar:
        tar.extractall(DATA_DIR)
    print("[extract] 완료")


# SEMHEAR_DIR이 다른 이름으로 풀렸을 경우 탐지
if not SEMHEAR_DIR.exists():
    alt = next((p for p in DATA_DIR.glob("Binaural*") if p.is_dir()), None)
    if alt:
        SEMHEAR_DIR = alt
        print(f"[info] SEMHEAR_DIR → {SEMHEAR_DIR}")

# SEMHEAR_DIR/FSD50K이 없을 때만 tar 추출 (tar의 FSD50K는 DCASE config와 다른 큐레이션
# 집합이므로, 이미 풀려 있으면 재추출해도 누락 파일은 채워지지 않음 → 전체 FSD50K는 Step 3c)
fsd_root = SEMHEAR_DIR / "FSD50K"
if not SEMHEAR_DIR.exists() or not fsd_root.exists():
    _extract_semhear_tar()
    if not SEMHEAR_DIR.exists():
        alt = next((p for p in DATA_DIR.glob("Binaural*") if p.is_dir()), None)
        if alt:
            SEMHEAR_DIR = alt
    fsd_root = SEMHEAR_DIR / "FSD50K"
else:
    print(f"[skip] 이미 압축 해제됨: {SEMHEAR_DIR}")

assert SEMHEAR_DIR.exists(), f"SEMHEAR_DIR not found: {SEMHEAR_DIR}"

# FSD50K 완전성 보고 (경고만) — 누락이 많으면 Step 3c로 전체 FSD50K 다운로드
if FSD50K_CONFIG.exists():
    fsd_missing = verify_dataset_complete(fsd_root, FSD50K_CONFIG, "FSD50K dev_audio", strict=False)
    if fsd_missing:
        print(f"[info] 전체 FSD50K가 필요하면 Step 3c 셀을 실행하세요 (누락 {len(fsd_missing)}개).")
else:
    print("[warn] FSD50K_config.json 없음 — Step 1(dev_set) 먼저 실행 필요. 완전성 검증 skip")

# 내부 구조 출력 (add_data.sh 디버깅용)
print(f"\nSEMHEAR_DIR: {SEMHEAR_DIR}")
print("내부 항목:")
for p in sorted(SEMHEAR_DIR.iterdir()):
    marker = "[dir]" if p.is_dir() else "[file]"
    print(f"  {marker} {p.name}")

print("\nStep 3 OK:", SEMHEAR_DIR)

## Step 3c. 전체 FSD50K dev_audio 다운로드 (선택 · 대용량)

Semantic Hearing tar의 FSD50K는 DCASE `FSD50K_config.json`과 **다른 큐레이션 집합**이라 요구 파일 일부가 없습니다.
요구 파일을 모두 채우려면 [Zenodo 4060432](https://zenodo.org/records/4060432)의 전체 `FSD50K.dev_audio`(분할 zip, ~24GB)를 받아 `SEMHEAR_DIR/FSD50K/`에 병합합니다.

> 시간/용량이 크므로 `RUN_FSD50K_DOWNLOAD = True`로 바꿔야 실행됩니다. 지금 당장은 Step 4의 config 필터링으로 있는 파일만으로 진행할 수 있습니다.

In [ ]:
# Step 3c: 전체 FSD50K dev_audio (Zenodo 4060432) 다운로드 → SEMHEAR_DIR/FSD50K 병합
# 대용량(~24GB)이므로 의도적으로 실행해야 함.
RUN_FSD50K_DOWNLOAD = False

fsd_root = SEMHEAR_DIR / "FSD50K"
FSD50K_CONFIG = DEV_SET_DIR / "config" / "FSD50K_config.json"

if not RUN_FSD50K_DOWNLOAD:
    print("[skip] RUN_FSD50K_DOWNLOAD=False — 전체 FSD50K 다운로드 건너뜀")
elif FSD50K_CONFIG.exists() and not missing_from_config(fsd_root, FSD50K_CONFIG):
    print("[skip] FSD50K 이미 완전 — 다운로드 불필요")
else:
    _ensure_archive_tools()
    fsd_dl = DOWNLOAD_DIR / "fsd50k"
    fsd_dl.mkdir(parents=True, exist_ok=True)
    ZENODO_FSD = "https://zenodo.org/record/4060432/files"
    parts = [f"FSD50K.dev_audio.z{i:02d}" for i in range(1, 6)] + ["FSD50K.dev_audio.zip"]
    for fname in parts:
        download_url(f"{ZENODO_FSD}/{fname}?download=1", fsd_dl / fname, label=fname)

    merged = fsd_dl / "FSD50K.dev_audio.unsplit.zip"
    if not merged.exists():
        print("[merge] zip -s 0")
        subprocess.run(
            ["zip", "-s", "0", "FSD50K.dev_audio.zip", "--out", merged.name],
            cwd=fsd_dl, check=True,
        )
    print(f"[extract] {merged.name} → {fsd_root}")
    fsd_root.mkdir(parents=True, exist_ok=True)
    _extract_with_unzip(merged, fsd_root)  # FSD50K.dev_audio/ 가 fsd_root 아래로 병합

    if FSD50K_CONFIG.exists():
        verify_dataset_complete(fsd_root, FSD50K_CONFIG, "FSD50K dev_audio", strict=True)
    print("Step 3c OK — 전체 FSD50K dev_audio 병합 완료")

## Step 3b. EARS 다운로드 (선택)

[EARS dataset](https://github.com/facebookresearch/ears_dataset) — speech 보강용 (p001 ~ p107, 각 ~수십MB).
전체 다운로드는 시간이 오래 걸립니다. `MAX_IDX`를 줄이면 일부만 받습니다.

In [ ]:
# Step 3b: EARS 다운로드
# 전체: MAX_IDX = 107 / 빠른 테스트: MAX_IDX = 3
MAX_IDX = 107

EARS_DIR = DATA_DIR / "EARS"
EARS_MARKER = EARS_DIR / ".ears_downloaded"

if EARS_MARKER.exists():
    print(f"[skip] EARS already downloaded: {EARS_DIR}")
else:
    EARS_DIR.mkdir(parents=True, exist_ok=True)
    BASE = "https://github.com/facebookresearch/ears_dataset/releases/download/dataset"
    for idx in range(1, MAX_IDX + 1):
        name = f"p{idx:03d}"
        zip_path = EARS_DIR / f"{name}.zip"
        person_dir = EARS_DIR / name
        if person_dir.exists() and any(person_dir.iterdir()):
            print(f"[skip] {name}")
            continue
        download_url(f"{BASE}/{name}.zip", zip_path, label=f"{name}.zip")
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(EARS_DIR)
        zip_path.unlink()
    EARS_MARKER.write_text("ok", encoding="utf-8")
    print(f"EARS 다운로드 완료: {EARS_DIR}")

print("Step 3b OK:", EARS_DIR)

## Step 4. add_data.sh 실행 — dev_set 보강

baseline의 `add_data.sh`를 Python에서 직접 호출합니다.
내부적으로 두 스크립트를 순서대로 실행합니다:

| 스크립트 | 입력 | 출력 | 내용 |
|----------|------|------|------|
| `add_interference.py` | `SEMHEAR_DIR` (bg set) | `dev_set/interference/` | Semantic Hearing non-target 클래스 → interference |
| `add_sound_event.py` | `SEMHEAR_DIR/FSD50K` + `EARS_DIR` | `dev_set/sound_event/` | FSD50K + EARS → target sound event 보강 |

> **참고**: baseline README 명령어
> ```bash
> bash add_data.sh --semhear_path /path/to/BinauralCuratedDataset --ears_path /path/to/EARS
> ```

In [ ]:
# Step 4: add_data.sh에 해당하는 Python 구현
# baseline: bash add_data.sh --semhear_path <SEMHEAR_DIR> --ears_path <EARS_DIR>

ADD_MARKER = DEV_SET_DIR / ".add_data_done"
EARS_DIR = DATA_DIR / "EARS"  # Step 3b에서 다운로드한 경로

COMMON = [
    "--target_sr=32000",
    "--amp_threshold=0.02",
    "--min_length=0.1",
    "--segment=10",
    "--shift=0.1",
]

if ADD_MARKER.exists():
    print("[skip] add_data already done")
else:
    # ── 1) add_interference.py : SEMHEAR/bg_scaper_fmt → interference ──
    int_marker = DEV_SET_DIR / ".interference_added"
    if int_marker.exists():
        print("[skip] interference already added")
    else:
        bg_dir = SEMHEAR_DIR / "bg_scaper_fmt"
        print(f"[run] add_interference.py --input_dir {bg_dir}")
        result = subprocess.run(
            [
                sys.executable, "add_interference.py",
                "--input_dir", str(bg_dir),
                "--output_dir", str(DEV_SET_DIR / "interference"),
                *COMMON,
            ],
            cwd=PROJECT_ROOT,
            capture_output=True, text=True,
        )
        print(result.stdout[-3000:] if result.stdout else "(no stdout)")
        if result.returncode != 0:
            print("[STDERR]", result.stderr[-2000:])
            raise RuntimeError("add_interference.py failed — stderr 확인")
        int_marker.write_text("ok", encoding="utf-8")
        print("[ok] interference added")

    # ── 2) add_sound_event.py : FSD50K (in SEMHEAR) + EARS → sound_event ──
    # pickup_json을 실제 존재하는 wav만 남기도록 필터링해서 호출 (전체 FSD50K를
    # Step 3c로 받았다면 필터링이 전부 통과 = 원본 config와 동일하게 동작).
    import json as _json

    def run_add_sound_event(input_dir: Path, pickup_json: Path):
        rows = _json.load(open(pickup_json))
        kept = [r for r in rows if (input_dir / r["file_name"]).exists()]
        print(f"[filter] {pickup_json.name}: {len(kept)}/{len(rows)} wav 존재")
        if not kept:
            print(f"[warn] {pickup_json.name}: 존재하는 wav 없음 — skip")
            return
        filtered = pickup_json.with_suffix(".filtered.json")
        filtered.write_text(_json.dumps(kept), encoding="utf-8")
        result = subprocess.run(
            [
                sys.executable, "add_sound_event.py",
                "--input_dir", str(input_dir),
                "--output_dir", str(DEV_SET_DIR / "sound_event"),
                "--pickup_json", str(filtered),
                *COMMON,
            ],
            cwd=PROJECT_ROOT, capture_output=True, text=True,
        )
        print(result.stdout[-3000:] if result.stdout else "(no stdout)")
        if result.returncode != 0:
            print("[STDERR]", result.stderr[-2000:])
            raise RuntimeError(f"add_sound_event.py ({input_dir.name}) failed — stderr 확인")

    se_marker = DEV_SET_DIR / ".sound_event_added"
    if se_marker.exists():
        print("[skip] sound_event already added")
    else:
        # 부분 실행으로 남은 출력은 add_sound_event.py의 target 중복 assert에 걸리므로 정리
        se_out = DEV_SET_DIR / "sound_event"
        if se_out.exists():
            shutil.rmtree(se_out)

        fsd = SEMHEAR_DIR / "FSD50K"
        ears_ok = EARS_DIR.exists() and any(EARS_DIR.iterdir())
        print(f"[info] FSD50K: {fsd} exists={fsd.exists()}")
        print(f"[info] EARS  : {EARS_DIR} exists={ears_ok}")

        print(f"[run] add_sound_event.py --input_dir {fsd.name}")
        run_add_sound_event(fsd, DEV_SET_DIR / "config" / "FSD50K_config.json")

        if ears_ok:
            print("[run] add_sound_event.py --input_dir EARS")
            run_add_sound_event(EARS_DIR, DEV_SET_DIR / "config" / "EARS_config.json")
        else:
            print("[warn] EARS 없음 — Step 3b 미실행. FSD50K만으로 진행.")

        se_marker.write_text("ok", encoding="utf-8")
        print("[ok] sound_event added")

    ADD_MARKER.write_text("ok", encoding="utf-8")

print("\nStep 4 OK")

## Step 5. 경로 검증 (dataloader 스mo크용)

In [ ]:
# 1) 원본 데이터셋 완전성 보고 (config 기준) — 경고만, 누락돼도 진행 (Step 3c로 보강 가능)
verify_dataset_complete(SEMHEAR_DIR / "FSD50K", DEV_SET_DIR / "config" / "FSD50K_config.json", "FSD50K dev_audio", strict=False)
EARS_DIR = DATA_DIR / "EARS"
EARS_CONFIG = DEV_SET_DIR / "config" / "EARS_config.json"
if EARS_DIR.exists() and any(EARS_DIR.iterdir()):
    verify_dataset_complete(EARS_DIR, EARS_CONFIG, "EARS", strict=False)
else:
    print("[warn] EARS 없음 — Step 3b 미실행")

# 2) dataloader 필수 경로 검증
REQUIRED = [
    "data/dev_set/sound_event/train", "data/dev_set/noise/train",
    "data/dev_set/interference/train", "data/dev_set/room_ir/train",
    "data/dev_set/metadata/valid.json",
    "src/modules/spatial_audio_synthesizer/spatial_audio_synthesizer.py",
]
missing = [r for r in REQUIRED if not (PROJECT_ROOT / r).exists()]
for r in REQUIRED:
    print(f"[{'OK' if r not in missing else 'MISSING'}] {r}")
if missing:
    print("\n[warn] missing paths — Step 7–8 skip 가능, unittest는 OK")
else:
    print("\nStep 5 OK")

---
## ★ unittest 전용 셀 (`test_new_dataset` + `test_new_datamodule`)

**dev set / Semantic Hearing / SpAudSyn 없이 실행 가능.**
Colab → Step 0 → helpers → **이 셀** 만으로도 검증 가능.

In [ ]:
def run_datamodule_tests(verbosity=2):
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    suite.addTests(loader.discover(
        start_dir=str(PROJECT_ROOT / "tests" / "datamodules"),
        pattern="test_new_*.py",
        top_level_dir=str(PROJECT_ROOT),
    ))
    stream = StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=verbosity).run(suite)
    print(stream.getvalue())
    print(f"\nRan {result.testsRun} | failures={len(result.failures)} | errors={len(result.errors)}")
    for label, items in [("FAILURES", result.failures), ("ERRORS", result.errors)]:
        if items:
            print(f"\n--- {label} ---")
            for test, tb in items:
                print(test, "\n", tb)
    return result

test_result = run_datamodule_tests()
assert not test_result.failures and not test_result.errors
print("\n★ unittest OK — new_dataset.py / new_datamodule.py (26 tests)")

['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender',

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender',

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## Step 7. Train DataModule 구성 (Step 1–2·5 완료 후)

In [ ]:
from src.datamodules.new_datamodule import DataModule

with open(PROJECT_ROOT / "config" / "label" / "m2dat_1c.yaml") as f:
    cfg = yaml.safe_load(f)
dm_args = cfg["datamodule"]["args"]
for split in ("train_dataloader", "val_dataloader"):
    if split in dm_args:
        ds = dm_args[split]["dataset"]
        ds["module"] = "src.datamodules.new_dataset"
        ds["main"] = "DatasetS3"

train_cfg = dm_args["train_dataloader"]
train_cfg.update(batch_size=2, num_workers=0, persistent_workers=False)
train_cfg["dataset"]["args"]["config"]["dataset_length"] = 8
val_cfg = dm_args.get("val_dataloader")
if val_cfg:
    val_cfg.update(batch_size=2, num_workers=0, persistent_workers=False)

dm = DataModule(train_dataloader=train_cfg, val_dataloader=val_cfg)
train_loader = dm.train_dataloader()
print("Step 7 OK | mode:", train_cfg["dataset"]["args"]["config"]["mode"])

## Step 8. Train / Val dataloader 스mo크 테스트

In [ ]:
batch = next(iter(train_loader))
describe_batch(batch, "train batch")
for key in ("mixture", "labels", "doas", "active"):
    assert key in batch and batch[key].dim() >= 2

# val은 valid split 데이터(sound_event/valid 등)가 있어야 동작.
# 부분 데이터로 train만 준비된 경우엔 skip하고 train 검증만 통과시킨다.
val_fg = PROJECT_ROOT / "data" / "dev_set" / "sound_event" / "valid"
if dm.val_dataset is not None and val_fg.exists():
    describe_batch(next(iter(dm.val_dataloader())), "val batch")
else:
    print(f"[skip] val dataloader — {val_fg} 없음 (valid split 미생성). train 검증만 수행.")

print("\nStep 8 OK — dataloader smoke test passed")